# Captcha CNN training on Colab
Set **Runtime > Change runtime type > T4 GPU** first, then run all cells top to bottom.
Needs `image_ftech_solver.zip` in the root of My Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp "/content/drive/MyDrive/image_ftech_solver.zip" /content/
!unzip -q -o /content/image_ftech_solver.zip -d /content/
!nvidia-smi -L
import glob
print(glob.glob("/content/image_ftech_solver/data/varun/1.png"))
print(glob.glob("/content/image_ftech_solver/v1/labels_vision.csv"))

In [ ]:
import csv, math, os, random, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

ROOT = "/content/image_ftech_solver"
LABELS_CSV = f"{ROOT}/v1/labels_vision.csv"
IMAGES = f"{ROOT}/data/varun"                 # raw originals, <id>.png
CACHE = "/content/raw_cache.npz"              # decoded images, so re-runs skip reading 120k PNGs
MODEL_PATH = "/content/drive/MyDrive/captcha_cnn.pt"   # saved to Drive so it survives the session
LENGTH = 5
SEED = 0

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then rerun from the top"
DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("device:", DEVICE, torch.cuda.get_device_name(0))

In [ ]:
with open(LABELS_CSV) as f:
    rows = [r for r in csv.DictReader(f) if r["source"] in ("vision", "human")]
assert all(len(r["label"]) == LENGTH for r in rows)

ALPHABET = "".join(sorted(set("".join(r["label"] for r in rows))))
CHAR_TO_IDX = {c: i for i, c in enumerate(ALPHABET)}

auto = [r for r in rows if r["source"] == "vision"]
gold = [r for r in rows if r["source"] == "human"]
random.Random(SEED).shuffle(auto)
n_train, n_test = int(0.7 * len(auto)), int(0.2 * len(auto))
split_rows = {
    "train": auto[:n_train],
    "test": auto[n_train:n_train + n_test],
    "val": auto[n_train + n_test:],
    "gold": gold,
}
print(f"alphabet ({len(ALPHABET)}): {ALPHABET}")
print({k: len(v) for k, v in split_rows.items()})

In [ ]:
all_ids = np.array(sorted({r["id"] for r in rows}, key=int))

def read_gray(i):
    return np.array(Image.open(os.path.join(IMAGES, f"{i}.png")).convert("L"))

cached = np.load(CACHE) if os.path.exists(CACHE) else None
if cached is not None and np.array_equal(cached["ids"], all_ids):
    imgs = cached["imgs"]
else:
    t = time.time()
    with ThreadPoolExecutor(8) as ex:
        imgs = np.stack(list(ex.map(read_gray, all_ids)))
    np.savez(CACHE, ids=all_ids, imgs=imgs)
    print(f"read {len(imgs)} images in {time.time() - t:.0f}s")
H, W = imgs.shape[1:]
print("images:", imgs.shape, imgs.dtype)

X = torch.from_numpy(imgs).to(DEVICE)   # uint8, all images on the GPU (~0.8 GB)
row_of = {i: k for k, i in enumerate(all_ids)}

def encode(label):
    return [CHAR_TO_IDX[c] for c in label]

def decode_pred(p):
    return "".join(ALPHABET[i] for i in p)

data = {
    name: (torch.tensor([row_of[r["id"]] for r in rs]), torch.tensor([encode(r["label"]) for r in rs]))
    for name, rs in split_rows.items()
}

In [ ]:
def to_input(x):
    """uint8 (B, H, W) -> float (B, 1, H, W) with ink = 1, paper = 0."""
    return (1 - x.float() / 255).unsqueeze(1)

def batches(split, batch_size, shuffle):
    idx, y = data[split]
    order = torch.randperm(len(idx)) if shuffle else torch.arange(len(idx))
    for k in range(0, len(idx), batch_size):
        j = order[k:k + batch_size]
        yield to_input(X[idx[j].to(DEVICE)]), y[j].to(DEVICE)

def augment(x, max_shift=4, max_rot=4):
    """Random shift (px) and rotation (degrees) per image, whole batch at once on the GPU.
    No flips: a mirrored letter is a different letter."""
    b, _, h, w = x.shape
    rnd = lambda: torch.rand(b, device=x.device) * 2 - 1
    ang = rnd() * max_rot * math.pi / 180
    cos, sin = torch.cos(ang), torch.sin(ang)
    tx, ty = rnd() * max_shift * 2 / w, rnd() * max_shift * 2 / h
    # rotation in normalised coordinates, corrected for the 150x45 aspect ratio
    theta = torch.stack([torch.stack([cos, -sin * h / w, tx], 1),
                         torch.stack([sin * w / h, cos, ty], 1)], 1)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, padding_mode="zeros", align_corners=False)

# sanity check: originals (top) vs augmented (bottom)
x, y = next(batches("train", 6, shuffle=True))
fig, ax = plt.subplots(2, 6, figsize=(15, 2.6))
for k, img in enumerate(torch.cat([x, augment(x)]).cpu()):
    ax[k // 6, k % 6].imshow(img[0], cmap="gray_r", vmin=0, vmax=1)
    ax[k // 6, k % 6].set_title(decode_pred(y[k % 6].tolist()))
    ax[k // 6, k % 6].axis("off")
plt.show()

In [ ]:
class CaptchaCNN(nn.Module):
    def __init__(self, n_classes, length=LENGTH, widths=(32, 64, 128, 192), dropout=0.3):
        super().__init__()
        layers, c_in = [], 1
        for c in widths:   # each block halves H and W: 45x150 -> 2x9
            layers += [
                nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            ]
            c_in = c
        self.features = nn.Sequential(*layers)
        n_cols = W // 2 ** len(widths)   # 9 columns left, kept apart so heads know left from right
        self.shared = nn.Sequential(nn.Flatten(), nn.Dropout(dropout),
                                    nn.Linear(c_in * n_cols, 512), nn.ReLU(inplace=True), nn.Dropout(dropout))
        self.heads = nn.ModuleList(nn.Linear(512, n_classes) for _ in range(length))

    def forward(self, x):
        z = self.features(x).mean(dim=2)   # average over height -> (B, C, columns)
        z = self.shared(z)
        return torch.stack([head(z) for head in self.heads], dim=1)   # (B, LENGTH, n_classes)

m = CaptchaCNN(len(ALPHABET))
print(f"{sum(p.numel() for p in m.parameters()) / 1e6:.2f}M parameters")

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def predict(model, split, batch_size=1024):
    model.eval()
    return torch.cat([model(x).argmax(-1).cpu() for x, _ in batches(split, batch_size, shuffle=False)])

def evaluate(model, split):
    p, y = predict(model, split), data[split][1]
    return {"exact": (p == y).all(1).float().mean().item(), "char": (p == y).float().mean().item()}

def train(cfg, tag=""):
    """Trains with cfg; keeps the epoch with the best val exact-match. Returns (model, best val)."""
    torch.manual_seed(cfg["seed"])
    model = CaptchaCNN(len(ALPHABET), dropout=cfg["dropout"]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    steps_per_epoch = math.ceil(len(data["train"][0]) / cfg["batch"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg["lr"], total_steps=cfg["epochs"] * steps_per_epoch)
    best, best_state, stale = -1.0, None, 0
    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        t, total = time.time(), torch.zeros((), device=DEVICE)
        bar = tqdm(batches("train", cfg["batch"], shuffle=True), total=steps_per_epoch,
                   desc=f"{tag}epoch {epoch}/{cfg['epochs']}", leave=False)
        for step, (x, y) in enumerate(bar, 1):
            if cfg["augment"]:
                x = augment(x)
            loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten(), label_smoothing=cfg["smoothing"])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            total += loss.detach()
            if step % 50 == 0:   # reading the loss forces a GPU sync, so only now and then
                bar.set_postfix(loss=f"{total.item() / step:.3f}")
        val = evaluate(model, "val")
        gold_r = evaluate(model, "gold")
        print(f"{tag}epoch {epoch:>2}/{cfg['epochs']}  loss {total.item() / steps_per_epoch:.3f}  "
              f"val exact {val['exact']:.2%}  char {val['char']:.2%}  "
              f"gold exact {gold_r['exact']:.2%}  {time.time() - t:.0f}s", flush=True)
        
        if val["exact"] > best:
            best, stale = val["exact"], 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= cfg["patience"]:
                print(f"{tag}stopping early, no val gain for {stale} epochs")
                break
    model.load_state_dict(best_state)
    return model, best

CFG = dict(epochs=25, batch=256, lr=2e-3, weight_decay=1e-4, dropout=0.3,
           smoothing=0.1, augment=True, patience=5, seed=SEED)

Optional: short hyperparameter search (5 runs x 4 epochs). Skip to go straight to training.

In [ ]:
# Short runs, one setting changed at a time. Copy the winner into CFG before cell 8.
# The first run prints how long an epoch takes; 5 runs x 4 epochs in total.
search = {"lr": [1e-3, 2e-3, 4e-3], "dropout": [0.2, 0.5]}
results = []
for name, values in search.items():
    for v in values:
        cfg = {**CFG, name: v, "epochs": 4, "patience": 2}
        _, val_exact = train(cfg, tag=f"[{name}={v}] ")
        results.append((name, v, val_exact))
        print(f"==> {name} = {v:<7} best val exact {val_exact:.2%}\n", flush=True)
print(sorted(results, key=lambda r: -r[2]))

Main training run

In [ ]:
model, best_val = train(CFG)
print(f"best val exact: {best_val:.2%}")

In [ ]:
# test = held-out Vision labels (agreement with Vision); gold = hand-checked (agreement with truth)
for split in ("val", "test", "gold"):
    r = evaluate(model, split)
    print(f"{split:<5} ({len(data[split][0]):>6} images)  exact {r['exact']:.2%}  char {r['char']:.2%}")

vision_gold = sum(r["raw"] == r["label"] for r in gold) / len(gold)
print(f"Vision raw read on gold, for comparison: {vision_gold:.2%} (before lookalike fixes)")

p_gold, y_gold = predict(model, "gold"), data["gold"][1]
print("per-position accuracy on gold:", [f"{a:.1%}" for a in (p_gold == y_gold).float().mean(0).tolist()])
mix = Counter((ALPHABET[t], ALPHABET[q]) for pp, yy in zip(p_gold, y_gold)
              for t, q in zip(yy.tolist(), pp.tolist()) if t != q)
print("most common confusions on gold (true -> predicted):", mix.most_common(10))

wrong = [k for k in range(len(gold)) if not torch.equal(p_gold[k], y_gold[k])]
cols = 4
rows_n = max(1, math.ceil(len(wrong) / cols))
fig, ax = plt.subplots(rows_n, cols, figsize=(16, 1.6 * rows_n), squeeze=False)
for a in ax.flat:
    a.axis("off")
for a, k in zip(ax.flat, wrong):
    a.imshow(imgs[row_of[gold[k]["id"]]], cmap="gray")
    a.set_title(f"{gold[k]['id']}  true {gold[k]['label']}  model {decode_pred(p_gold[k].tolist())}", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
torch.save({
    "state_dict": model.state_dict(),
    "alphabet": ALPHABET,
    "length": LENGTH,
    "image_size": (H, W),
    "widths": (32, 64, 128, 192),
    "config": CFG,
    "input": "grayscale HxW, x = 1 - pixel/255 (ink = 1), shape (B, 1, H, W)",
    "scores": {s: evaluate(model, s) for s in ("val", "test", "gold")},
}, MODEL_PATH)

def load_model(path=MODEL_PATH):
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    m = CaptchaCNN(len(ckpt["alphabet"]), ckpt["length"], ckpt["widths"], ckpt["config"]["dropout"]).to(DEVICE)
    m.load_state_dict(ckpt["state_dict"])
    return m.eval(), ckpt

reloaded, ckpt = load_model()
assert torch.equal(predict(reloaded, "gold"), predict(model, "gold")), "reloaded model predicts differently"
print(f"saved {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1e6:.1f} MB), reload OK:", ckpt["scores"])

In [ ]:
@torch.no_grad()
def read_captcha(path, model=reloaded, alphabet=ckpt["alphabet"]):
    x = torch.from_numpy(np.array(Image.open(path).convert("L")))[None].to(DEVICE)
    probs = model(to_input(x)).softmax(-1)[0]           # (LENGTH, n_classes)
    conf, idx = probs.max(-1)
    return "".join(alphabet[i] for i in idx.tolist()), conf.min().item()   # weakest character's confidence

for i in ("57710", "94667", "87563"):
    print(i, read_captcha(os.path.join(IMAGES, f"{i}.png")))

Optional: write review queue to Drive

In [ ]:
@torch.no_grad()
def model_reads(ids, batch_size=1024):
    reads, confs = [], []
    for k in range(0, len(ids), batch_size):
        x = X[torch.tensor([row_of[i] for i in ids[k:k + batch_size]]).to(DEVICE)]
        probs = model(to_input(x)).softmax(-1)
        conf, idx = probs.max(-1)
        reads += ["".join(ALPHABET[i] for i in p) for p in idx.cpu().tolist()]
        confs += conf.min(-1).values.cpu().tolist()
    return reads, confs

model.eval()
pool = list(auto)   # Vision-labeled only; gold stays untouched as the final test
reads, confs = model_reads([r["id"] for r in pool])
queue = sorted(((r["id"], r["label"], m, c) for r, m, c in zip(pool, reads, confs) if m != r["label"]),
               key=lambda t: -t[3])
with open("/content/drive/MyDrive/review_queue.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "vision_label", "model_read", "model_confidence"])
    w.writerows(queue)
print(f"{len(queue)} of {len(pool)} Vision labels disagree with the model -> review_queue.csv")
print(queue[:10])